# 1.13 第二次 backward 為什麼梯度變大？


你按同一個按鈕兩次，本來想得到同樣結果，卻發現記錄的數字翻倍。不是參數突然變得更敏感，而是軟體把第二次結果加在第一次上了。PyTorch 的 `.grad` 就像一個收集桶：每次 `backward()` 把新梯度倒進去，不會自動先清空。若我們想每輪只按當前這批資料更新，就要把上輪的桶清掉。

先回顧 [1.10](https://birdhackor.github.io/tiny-perceptron-vlm/1.10.html) 的前向計算與求導：運算建立關係，`backward` 再往回算敏感度。這裡用 w=2、代價 w²，梯度是 `2*w=4`。不更新 w，再建立一次同樣代價並求導，新的梯度仍是 4；但 `.grad` 存放的是舊4+新4=8。這種累加是設計的一部分，後面可利用它合併多小份資料的梯度；忘記它才會成為錯誤。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

w = nn.Parameter(torch.tensor(2.0))
(w.square()).backward()
first = w.grad.clone()
(w.square()).backward()
second = w.grad.clone()
w.grad = None
(w.square()).backward()
print(first.item(), second.item(), w.grad.item(), w.item())
assert second.item() == 2 * first.item()

`nn.Parameter` 把 tensor 標成模型的可調參數，並預設追蹤梯度。輸入 w=2。每行 `w.square()` 都重新做一次前向計算，再對這次結果求導。`clone()` 把當時梯度複製一份，讓 `first`、`second` 儲存當時的數字，而不是以後跟著桶裡的內容變。輸出應為 `4,8,4,2`：前兩次記錄4與8，清除後第三次又只存4，參數始終2。

`w.grad=None` 表示把收集桶清空，下次求導會重新建立梯度。也可將現有梯度設為零；正式使用最佳化器時通常呼叫 `optimizer.zero_grad()`，這會對它負責的參數一起清除。最佳化器（optimizer）是按更新規則管理參數的工具，本例還沒有建立它。`backward` 只負責求導和累加，最佳化器的 `step()` 才會動參數。

要分清另一個不同問題：對同一個 `loss` 物件連續呼叫兩次 `backward`，可能出現「計算圖已經釋放」的錯誤。PyTorch 預設第一次求導後釋放部分中間材料；本例每次重新算 `w.square()`，所以用的是新關係，不需要保留舊圖。累加發生在可調參數 w 自己的 `.grad`，不需要把舊前向圖保留到下一輪。

通常一輪訓練依序是清除舊梯度、算新預測與代價、求導、更新。刻意跨小份累積時則會改變清除與更新的時機，第 [16.6](https://birdhackor.github.io/tiny-perceptron-vlm/16.6.html) 節會展開。現在先把「新的敏感度」與「桶裡累積的總和」分開讀。

練習只註釋掉 `w.grad = None`，在行首加 `#` 就會讓它不執行。先預測第三次記錄應為12、參數仍2，再跑核對。最後的檢查仍只比較前兩次，所以會通過；要判斷清除是否正確，還得看第三個輸出。這提醒我們，某個檢查通過只說明它檢查的那件事。
